# Критерии оценки ДЗ 

> **Версия LanceDB: `lancedb<0.20` !!!**

Делаете прогоны на test_dataset, выбиваете максимум по метрикам. Я буду проверять на eval_dataset

---

## Базовые требования (4 балла)

| # | Критерий | Баллы |
|---|----------|-------|
| 1 | Описан процесс подготовки данных (`data_preporation.py`, `embedder.py`) | 1 |
| 2 | Описаны методы/функции для работы с векторной БД в `lance_db.py` (обязательно `read_all`) | 1 |
| 3 | Данные записаны в БД и можно выполнить `read_all` | 1 |
| 4 | Описан флоу retrieve и agent (`agent.py`, `retriever.py`) | 1 |

---

## Метрики качества (до 9 баллов)

### F1@K

| Порог    | Баллы |
|----------|-------|
| > 0.50   | 1     |
| > 0.75   | 2     |
| > 0.90   | 3     |

### NDCG@K

| Порог    | Баллы |
|----------|-------|
| > 0.65   | 1     |
| > 0.75   | 2     |
| > 0.90   | 3     |

### Faithfulness

| Порог    | Баллы |
|----------|-------|
| > 0.75   | 1     |
| > 0.85   | 2     |
| > 0.95   | 3     |

---

## Лог эволюции (до 2 баллов)

Оценивается индивидуально на основании полноты информации в `EVOLUTION.md`.

---

## Штрафы

| Условие | Штраф |
|---------|-------|
| Суммарный объём всех передаваемых для генерации чанков > 6000 символов | −100500 баллов |
| Использовали дополнительные библиотеки и не положили requirements.txt | −100500 баллов |

---

**Максимум: 15 баллов** (4 базовых + 9 метрики + 2 эволюция)

# Валидация RAG-агента через модуль `evaluation.py`

Демонстрация полного пайплайна оценки:
1. Подключаемся к векторному хранилищу и создаём агента
2. Загружаем тестовый датасет
3. Прогоняем агента на всех вопросах (многопоточно)
4. Вызываем `evaluate()` — получаем DataFrame с 7 метриками
5. Анализируем результаты

---
## 0. Подготовка

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import settings
from openai import OpenAI

In [2]:
client = OpenAI(
    base_url="https://api.polza.ai/api/v1",
    api_key=settings.polza_ai_api_key,
)

MODEL = "openai/gpt-4o-mini"

---
## 1. Подключаемся к векторному хранилищу и создаём агента

In [3]:
import lancedb

VECTORSTORE_DIR = "lance_db/vectorstore"
TABLE_NAME = "chunks"

db = lancedb.connect(VECTORSTORE_DIR)
table = db.open_table(TABLE_NAME)

print(f"\u2705 Таблица '{TABLE_NAME}' открыта: {table.count_rows()} строк")

✅ Таблица 'chunks' открыта: 47 строк


In [4]:
from models import RAGAgentAnswer
from agent import RAGAgent
from embedder import Embedder


embedder = Embedder(api_key=client.api_key, base_url=str(client.base_url))
agent = RAGAgent(client=client, model=MODEL, table=table, embedder=embedder)

print("\u2705 Агент готов")

✅ Агент готов


---
## 2. Загрузка тестового датасета

Датасет: `data/test_dataframe.xlsx` — 150 вопросов с ground truth.

| Колонка | Описание |
|---------|----------|
| `id` | Уникальный идентификатор вопроса |
| `question` | Вопрос на естественном языке |
| `ground_truth_chunks` | Эталонные чанки (строковое представление списка) |
| `document` | Год отчёта (2019 / 2025) |

In [5]:
df_test = pd.read_excel("dataset/test_dataframe.xlsx")
print(f"Размер: {df_test.shape}")
print(f"Колонки: {list(df_test.columns)}")
# df_test = df_test.head(20)
df_test.head(3)

Размер: (150, 4)
Колонки: ['id', 'question', 'relevant_text', 'document']


,id,question,relevant_text,document
0,e876ec3b-7442-4145-99d0-d6bb85fd982d,В 2019 году как изменился индекс потребительск...,['Как мы предполагали при анализе результатов ...,2019
1,c7304cf4-9ebf-461b-acce-4dc987c21bcc,"Согласно данным 2025 года, какой фактор являет...","['Одним из главных негативных факторов, тормоз...",2025
2,7e727f79-c6ff-4ff0-b101-bcec2b0f7260,"Согласно данным 2019 года, что является основн...","['Очевидно, что основным фактором, определяющи...",2019


---
## 3. Прогон агента на тестовом датасете

Запускаем агента на каждом вопросе и сохраняем `list[RAGAgentAnswer]`.  
Это **дорогая операция** (150 вопросов × N API-вызовов), поэтому используем многопоточность.

In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def _run_single(idx: int, row_id: str, question: str, agent: RAGAgent) -> tuple[int, RAGAgentAnswer]:
    """Один вызов агента (для параллельного запуска)."""
    try:
        result = agent.run(question, verbose=False, dataset_row_id=row_id)
    except Exception as e:
        print(f"  [Ошибка] вопрос {idx}: {e}")
        result = RAGAgentAnswer(dataset_row_id=row_id, answer=f"[Ошибка: {e}]", retrieved_chunks=None)
    return idx, result


def run_evaluation(df: pd.DataFrame, agent: RAGAgent, max_workers: int = 4) -> list[RAGAgentAnswer]:
    """Прогнать агента на всех вопросах (многопоточно)."""
    results: list[RAGAgentAnswer | None] = [None] * len(df)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_run_single, idx, row["id"], row["question"], agent): idx for idx, row in df.iterrows()
        }
        for future in tqdm(as_completed(futures), total=len(futures), desc="Прогон агента"):
            idx, result = future.result()
            results[idx] = result

    return results

In [7]:
results = run_evaluation(df_test, agent)

print(f"\nВсего результатов: {len(results)}")
print(f"С чанками: {sum(1 for r in results if r.retrieved_chunks)}")
print(f"Без чанков (None): {sum(1 for r in results if not r.retrieved_chunks)}")

Прогон агента: 100%|██████████| 150/150 [01:10<00:00,  2.14it/s]


Всего результатов: 150
С чанками: 147
Без чанков (None): 3


---
## 4. Оценка через `evaluation.evaluate()`

Весь расчёт метрик — **одна строка**. Модуль `evaluation.py` принимает:
- `answers`: `list[RAGAgentAnswer]` — ответы агента
- `dataset`: `pd.DataFrame` — тестовый датасет с ground truth
- `client` + `model` — для вычисления Faithfulness (LLM-as-judge)

Возвращает `pd.DataFrame` с метриками по каждому вопросу.

In [8]:
df_test

,id,question,relevant_text,document
0,e876ec3b-7442-4145-99d0-d6bb85fd982d,В 2019 году как изменился индекс потребительск...,['Как мы предполагали при анализе результатов ...,2019
1,c7304cf4-9ebf-461b-acce-4dc987c21bcc,"Согласно данным 2025 года, какой фактор являет...","['Одним из главных негативных факторов, тормоз...",2025
2,7e727f79-c6ff-4ff0-b101-bcec2b0f7260,"Согласно данным 2019 года, что является основн...","['Очевидно, что основным фактором, определяющи...",2019
3,d52e6df3-d5ad-404e-9da9-f6f1855d16df,Какой процент респондентов в 2025 году затрудн...,"['затрудняюсь ответить — 0,6%']",2025
4,536bf426-d85f-472c-b434-6796078cf647,"Согласно данным 2019 года, какой был баланс от...","['В результате, баланс отрицательных и положит...",2019
...,...,...,...,...
145,17fc34cc-b8eb-49f0-b384-1d1db1998846,"Согласно данным 2025 года, какой процент респо...","['- затрудняюсь ответить — 0,3%']",2025
146,60a79f64-8f20-44dc-955e-8e8deee332e8,обобщающий индекс потребительской уверенности ...,['Обобщающий индекс потребительской уверенност...,2025
147,f5995543-80ec-4eb6-a0e2-fdf0b5d0de05,В 2025 году что является важнейшим конъюнктурн...,['Именно ожидания людей и бизнеса являются важ...,2025
148,635706a4-8be4-4550-9782-02bbaf29e2cd,Как изменилось восприятие респондентов относит...,['аспределение мнений респондентов сместилось ...,2019


In [9]:
results

[RAGAgentAnswer(dataset_row_id='e876ec3b-7442-4145-99d0-d6bb85fd982d', answer='Информация не найдена.', retrieved_chunks=[Chunk(text='. Индекс потребительской уверенности Росстата является важнейшей компонентой сводного индекса экономического настроения (ИЭН ВШЭ), который ежеквартально рассчитывается Центром конъюнктурных исследований и интегрально характеризует состояние делового климата в экономике страны. Основные итоги II квартала 2019 года − Индекс потребительской уверенности (ИПУ) повысился на 1 процентный пункт (п. п.) до значения ( -15)% − Улучшени е личного материального положения за последние 12 месяцев констатировали 11% респондентов, а его ухудшени е – 32% (кварталом ранее – 10 и 34%) − Ухудшения личного материального положения в течение следующих 12 месяцев ожидают 19% респондентов (кварталом ранее - 23%) − Позитивно оценили произошедшие за год изменения в экономике России 12% участников опроса, негативно – 42% (кварталом ранее – 11 и 45%) − Положительных изменений в эконо

In [10]:
from evaluation import evaluate

df_metrics = evaluate(
    answers=results,
    dataset=df_test,
    client=client,
    model=MODEL,
    overlap_threshold=0.1,
    compute_faithfulness=True,
)

print(f"\u2705 Оценка завершена: {df_metrics.shape[0]} строк, {df_metrics.shape[1]} колонок")
df_metrics.head(10)

Faithfulness (RAGAS): 100%|██████████| 147/147 [05:21<00:00,  2.19s/it]

✅ Оценка завершена: 150 строк, 9 колонок


,id,question,precision,recall,f1,mrr,map,ndcg,faithfulness
0,e876ec3b-7442-4145-99d0-d6bb85fd982d,В 2019 году как изменился индекс потребительск...,0.5,1.0,0.666667,1.0,1.0,1.00000,0.000000
1,c7304cf4-9ebf-461b-acce-4dc987c21bcc,"Согласно данным 2025 года, какой фактор являет...",0.5,1.0,0.666667,1.0,1.0,1.00000,0.750000
2,7e727f79-c6ff-4ff0-b101-bcec2b0f7260,"Согласно данным 2019 года, что является основн...",0.5,1.0,0.666667,1.0,1.0,1.00000,0.333333
3,d52e6df3-d5ad-404e-9da9-f6f1855d16df,Какой процент респондентов в 2025 году затрудн...,0.5,1.0,0.666667,1.0,1.0,1.00000,0.000000
4,536bf426-d85f-472c-b434-6796078cf647,"Согласно данным 2019 года, какой был баланс от...",0.5,1.0,0.666667,1.0,1.0,1.00000,1.000000
5,e3453b4c-2d2a-4569-899d-f731158e2d36,"Согласно данным 2025 года, какие социальные по...",0.5,1.0,0.666667,1.0,1.0,1.00000,0.000000
6,a3ae1b1c-a296-4e85-b8df-6ca1ed2093d0,В 2025 году какова связь между потребительским...,0.5,1.0,0.666667,1.0,1.0,1.00000,0.000000
7,25a2eff0-2893-471f-a145-20aa63633e13,В 2019 году какое место в рейтинге стран ЕС по...,0.5,1.0,0.666667,1.0,1.0,1.00000,1.000000
8,49287b9e-2dad-4902-a389-bc0a34494dfc,Какая доля населения в 2025 году в основном фо...,0.5,1.0,0.666667,1.0,1.0,1.00000,0.666667
9,6192474e-db88-47a1-9c27-5c5e89d5e430,Какой процент респондентов оказался в затрудни...,0.5,1.0,0.666667,0.5,0.5,0.63093,0.000000


---
## 5. Анализ результатов

In [12]:
metric_cols = ["precision", "recall", "f1", "mrr", "map", "ndcg", "faithfulness"]
display_names = ["Precision@K", "Recall@K", "F1@K", "MRR", "MAP", "NDCG@K", "Faithfulness"]

summary = df_metrics[metric_cols].mean()
summary.index = display_names

print("=" * 45)
print("  ИТОГОВЫЕ МЕТРИКИ КАЧЕСТВА RAG-АГЕНТА")
print("=" * 45)
for name, val in summary.items():
    bar = "\u2588" * int(val * 30) + "\u2591" * (30 - int(val * 30))
    print(f"  {name:<15} {val:.4f}  {bar}")
print("=" * 45)

  ИТОГОВЫЕ МЕТРИКИ КАЧЕСТВА RAG-АГЕНТА
  Precision@K     0.5333  ████████████████░░░░░░░░░░░░░░
  Recall@K        0.9644  ████████████████████████████░░
  F1@K            0.6722  ████████████████████░░░░░░░░░░
  MRR             0.9633  ████████████████████████████░░
  MAP             0.9478  ████████████████████████████░░
  NDCG@K          0.9563  ████████████████████████████░░
  Faithfulness    0.6211  ██████████████████░░░░░░░░░░░░


In [13]:
# Сводная таблица
df_summary = pd.DataFrame(
    list(zip(display_names, summary.values)),
    columns=["Метрика", "Значение"],
)
df_summary["Значение"] = df_summary["Значение"].round(4)
df_summary

,Метрика,Значение
0,Precision@K,0.5333
1,Recall@K,0.9644
2,F1@K,0.6722
3,MRR,0.9633
4,MAP,0.9478
5,NDCG@K,0.9563
6,Faithfulness,0.6211


In [14]:
# Распределение метрик
df_metrics[metric_cols].describe().round(4)

,precision,recall,f1,mrr,map,ndcg,faithfulness
count,150.0000,150.0000,150.0000,150.0000,150.0000,150.0000,150.0000
mean,0.5333,0.9644,0.6722,0.9633,0.9478,0.9563,0.6211
std,0.1604,0.1599,0.1221,0.1648,0.1802,0.1629,0.4179
min,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,0.5000,1.0000,0.6667,1.0000,1.0000,1.0000,0.0000
50%,0.5000,1.0000,0.6667,1.0000,1.0000,1.0000,0.7500
75%,0.5000,1.0000,0.6667,1.0000,1.0000,1.0000,1.0000
max,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000


In [15]:
# Худшие вопросы по F1
worst = df_metrics.nsmallest(5, "f1")[["id", "question", "precision", "recall", "f1", "faithfulness"]]
worst

,id,question,precision,recall,f1,faithfulness
106,29a8ce92-1b66-4b9e-9a30-7736518f1cb3,Какие шкалы измерения используются для предста...,0.0,0.000000,0.000000,0.0
131,655c0069-08e7-4a99-b5c4-e2fcbddfa716,В 2019 году какие меры мог применить ЦБ РФ с 1...,0.0,0.000000,0.000000,0.0
139,174f1600-5386-49d9-b2ba-217af4b2736f,"В 2025 году какие годы были отмечены как ""тучн...",0.0,0.000000,0.000000,0.0
53,58afd835-8e1c-43e8-8290-c29546b2378c,Как изменения в уровне бедности и дифференциац...,1.0,0.333333,0.500000,0.0
0,e876ec3b-7442-4145-99d0-d6bb85fd982d,В 2019 году как изменился индекс потребительск...,0.5,1.000000,0.666667,0.0


In [16]:
# Лучшие вопросы по F1
best = df_metrics.nlargest(5, "f1")[["id", "question", "precision", "recall", "f1", "faithfulness"]]
best

,id,question,precision,recall,f1,faithfulness
12,96502955-0a6f-45f6-8064-ce293bbf8d47,Какие факторы могут повлиять на снижение индек...,1.0,1.0,1.0,0.000000
33,91bc2a46-4eb8-4d9e-919b-9b85ef2fa1fa,Как изменились условия для крупных покупок и с...,1.0,1.0,1.0,1.000000
34,543fd8db-02a7-48ae-a311-fa78bbf1c8f3,В 2025 году как изменилась динамика индекса по...,1.0,1.0,1.0,1.000000
46,81b854bd-5b3c-4b43-ae80-c9755f76e36e,Какой уровень потребительской уверенности в Ро...,1.0,1.0,1.0,0.833333
48,59e9f7f3-5764-4af1-b7ba-7b22b4c0143f,Как изменения в инфляции и доходах населения в...,1.0,1.0,1.0,0.000000
